# FHB Severity Estimation — Logistic Regression Classifier

**Workflow:**
1. Load `Pixels_All.xlsx` and train a logistic regression classifier with stratified k-fold cross-validation
2. Load YOLO detection model + SAM segmentation models
3. Loop over all images in a directory:
   - Detect wheat heads with YOLO → segment each head with SAM
   - For each head mask, extract RGB pixels and classify with the trained logistic regression model
   - Compute per-mask FHB severity (% pixels classified as Infected)
   - Average severity across all heads in the image
4. Save results to CSV

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt

from ultralytics import YOLO, SAM, RTDETR
from mobile_sam import sam_model_registry, SamPredictor

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report

np.random.seed(42)

## 1. Configuration — set paths here

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ROOT_DIR       = #path to images    # folder of images to process
OUTPUT_CSV     = #output directory

YOLO_WEIGHTS   = #path to YOLO model weights
Mobile_SAM_CHECKPOINT = #path to Mobile SAM model weights
SAM3_CHECKPOINT = #path to SAM3 model weights

PIXEL_VALUES_XLSX = 'Data\Pixels_All.xlsx'

# ── Logistic Regression / CV settings ─────────────────────────────────────
N_FOLDS  = 5      # number of stratified k-folds for cross-validation
LR_C     = 1.0    # inverse regularisation strength (higher = less regularisation)

## 2. Train Logistic Regression on Pixel Values with K-Fold Cross-Validation

In [ ]:
df_vals = pd.read_excel(PIXEL_VALUES_XLSX, sheet_name='Sheet1')
print(f'Pixel Values shape: {df_vals.shape}')
print('Class balance:')
print(df_vals['Cluster'].value_counts().rename({0: 'Healthy', 1: 'Infected'}))

X = df_vals[['R', 'G', 'B']].values.astype(np.float64)
y = df_vals['Cluster'].values

# Pipeline: standardise then logistic regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(C=LR_C, max_iter=1000, random_state=42)),
])

# Cross-validation
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
cv_results = cross_validate(
    pipeline, X, y, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=False
)

print(f'\n=== {N_FOLDS}-Fold Cross-Validation Results ===')
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    scores = cv_results[f'test_{metric}']
    print(f'  {metric:>10}: {scores.mean():.4f} ± {scores.std():.4f}')

# Fit final model on all training data
pipeline.fit(X, y)
print('\nFinal model trained on full Pixel Values dataset.')
print(classification_report(y, pipeline.predict(X), target_names=['Healthy', 'Infected']))

# Show coefficients
coef_df = pd.DataFrame({
    'Channel':     ['R', 'G', 'B'],
    'Coefficient': pipeline.named_steps['logreg'].coef_[0]
})
print('\nLogistic Regression Coefficients (standardised features):')
print(coef_df.to_string(index=False))

## 3. Load Detection and Segmentation Models

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

yolo_model = YOLO(YOLO_WEIGHTS)

mobile_sam = sam_model_registry['vit_t'](checkpoint=Mobile_SAM_CHECKPOINT)
mobile_sam.to(device=device)
mobile_sam.eval()
predictor = SamPredictor(mobile_sam)

sam3_model = SAM(SAM3_CHECKPOINT)

print('Models loaded.')

## 4. Main Processing Loop - MobileSAM Segmentation

In [ ]:
records = []
image_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

for dirpath, _, filenames in os.walk(ROOT_DIR):
    for filename in sorted(filenames):
        if not filename.lower().endswith(image_extensions):
            continue

        img_path = os.path.join(dirpath, filename)
        print(f'Processing: {img_path}', end=' ... ')

        # ── Load image ────────────────────────────────────────────────────
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print('SKIPPED (unreadable)')
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # ── YOLO detection ────────────────────────────────────────────────
        yolo_results = yolo_model(img_path, verbose=False)
        boxes = []
        for result in yolo_results:
            for box in result.boxes.xyxy:
                boxes.append(box.tolist())

        if len(boxes) == 0:
            print('SKIPPED (no detections)')
            records.append({'Image': filename, 'Path': img_path,
                            'Num_Heads': 0, 'Mean_Severity_Pct': np.nan})
            continue

        # ── SAM segmentation ─────────────────────────────────────────────
        predictor.set_image(img_rgb)

        per_head_severities = []
        for box in boxes:
            input_box = np.array(box)
            masks, _, _ = predictor.predict(
                point_coords=None,
                point_labels=None,
                box=input_box[None, :],
                multimask_output=False,
            )
            mask = masks[0].astype(bool)

            # ── Logistic Regression classification ────────────────────────
            pixels = img_rgb[mask].astype(np.float64)
            if len(pixels) == 0:
                continue 
            labels = pipeline.predict(pixels)
            severity = float(np.sum(labels == 1) / len(labels) * 100)
            per_head_severities.append(severity)

        mean_severity = float(np.mean(per_head_severities))

        print(f'{len(per_head_severities)} heads | mean severity = {mean_severity:.2f}%')

        records.append({
            'Image':             filename,
            'Path':              img_path,
            'Num_Heads':         len(per_head_severities),
            'Mean_Severity_Pct': mean_severity,
        })

# ── Save results ─────────────────────────────────────────────────────────
df_results = pd.DataFrame(records)
df_results.to_csv(OUTPUT_CSV, index=False)
print(f'\nDone. Results saved to: {OUTPUT_CSV}')
df_results.head(10)

## 5. Quick Summary

In [ ]:
print('=== Results Summary ===')
print(f'Images processed : {len(df_results)}')
print(f'Images with heads: {df_results["Num_Heads"].gt(0).sum()}')
print(f'Mean severity    : {df_results["Mean_Severity_Pct"].mean():.2f}%')
print(f'Std severity     : {df_results["Mean_Severity_Pct"].std():.2f}%')
print()
print(df_results[['Image', 'Num_Heads', 'Mean_Severity_Pct']]
      .sort_values('Mean_Severity_Pct', ascending=False)
      .to_string(index=False))

valid = df_results['Mean_Severity_Pct'].dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid, bins=20, edgecolor='black', color='#E53935', alpha=0.8)
ax.set_xlabel('Mean FHB Severity per Image (%)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Logistic Regression FHB Severity Estimates')
plt.tight_layout()
plt.show()

## 6. Alternative Processing Loop — SAM3 Segmentation

Same as the loop above, but segments each detected head with **SAM3** instead of MobileSAM (all boxes for an image are segmented in a single batched call, rather than looping per box). Requires `sam3_model` from section 3 and `pipeline` from section 2.

In [ ]:
records_sam3 = []
image_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

for dirpath, _, filenames in os.walk(ROOT_DIR):
    for filename in sorted(filenames):
        if not filename.lower().endswith(image_extensions):
            continue

        img_path = os.path.join(dirpath, filename)
        print(f'Processing: {img_path}', end=' ... ')

        # ── Load image ────────────────────────────────────────────────────
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print('SKIPPED (unreadable)')
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # ── YOLO detection ────────────────────────────────────────────────
        yolo_results = yolo_model(img_path, verbose=False, max_det=800)
        boxes = []
        for result in yolo_results:
            for box in result.boxes.xyxy:
                boxes.append(box.tolist())

        if len(boxes) == 0:
            print('SKIPPED (no detections)')
            records_sam3.append({'Image': filename, 'Path': img_path,
                            'Num_Heads': 0, 'Mean_Severity_Pct': np.nan})
            continue

        # ── SAM3 segmentation (all boxes in a single batched call) ───────
        # Keep the mask stack on-device and convert one mask at a time below —
        # materialising the full (N, H, W) array in one go can exceed available
        # RAM on images with many detected heads at full resolution.
        sam3_results = sam3_model(img_path, bboxes=boxes, verbose=False)
        masks_tensor = sam3_results[0].masks.data

        per_head_severities = []
        for i in range(masks_tensor.shape[0]):
            mask = masks_tensor[i].cpu().numpy().astype(bool)

            # ── Logistic Regression classification ────────────────────────
            pixels = img_rgb[mask].astype(np.float64)
            if len(pixels) == 0:
                continue
            labels = pipeline.predict(pixels)
            severity = float(np.sum(labels == 1) / len(labels) * 100)
            per_head_severities.append(severity)

        del sam3_results, masks_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        mean_severity = float(np.mean(per_head_severities))

        print(f'{len(per_head_severities)} heads | mean severity = {mean_severity:.2f}%')

        records_sam3.append({
            'Image':             filename,
            'Path':              img_path,
            'Num_Heads':         len(per_head_severities),
            'Mean_Severity_Pct': mean_severity,
        })

# ── Save results ─────────────────────────────────────────────────────────
df_results_sam3 = pd.DataFrame(records_sam3)
df_results_sam3.to_csv(OUTPUT_CSV, index=False)
print(f'\nDone. Results saved to: {OUTPUT_CSV}')
df_results_sam3.head(10)